# 05 — Export to TFLite

Convert trained Keras model to TFLite with float16 quantization.
Validate that TFLite output matches Keras predictions.

## What this does
1. Loads trained Keras model
2. Converts to TFLite with float16 quantization (~50% size reduction)
3. Validates output matches Keras (max deviation < 0.001)
4. Exports TypeScript test fixtures (JSON) for app-side validation
5. Copies TFLite to React Native assets directory

## Inputs
```
models/pose_correction.keras   — trained model
models/test_fixtures.npz       — test data from notebook 04
```

## Outputs
```
models/exported/pose_correction.tflite     — quantized model (~60KB)
models/exported/ts_test_fixtures.json      — TypeScript validation data
../../apps/mobile/assets/models/pose_correction.tflite — copy for React Native
```

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import tensorflow as tf
from pathlib import Path
import json

print(f'TensorFlow version: {tf.__version__}')

## Load trained model

In [ ]:
model = tf.keras.models.load_model('../models/pose_correction.keras')
model.summary()
print(f'\nTotal parameters: {model.count_params():,}')

## Convert to TFLite with float16 quantization

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

output_dir = Path('../models/exported')
output_dir.mkdir(parents=True, exist_ok=True)
tflite_path = output_dir / 'pose_correction.tflite'
tflite_path.write_bytes(tflite_model)

size_kb = len(tflite_model) / 1024
print(f'TFLite model saved: {tflite_path}')
print(f'Model size: {size_kb:.1f} KB')

## Validate TFLite output matches Keras

Max deviation should be < 0.001 for float16 quantization.

In [ ]:
# Load test fixtures from training
fixtures = np.load('../models/test_fixtures.npz')
test_inputs = fixtures['inputs']
keras_phase = fixtures['phase_expected']
keras_correction = fixtures['correction_expected']

# Set up TFLite interpreter
interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('TFLite model info:')
print(f'  Input: {input_details[0]["shape"]} ({input_details[0]["dtype"]})')
for od in output_details:
    print(f'  Output "{od["name"]}": {od["shape"]} ({od["dtype"]})')

In [ ]:
max_phase_dev = 0
max_correction_dev = 0

for i in range(len(test_inputs)):
    input_data = test_inputs[i:i+1].astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    
    outputs = {}
    for od in output_details:
        tensor = interpreter.get_tensor(od['index'])
        if tensor.shape[-1] == 1:
            outputs['phase'] = tensor
        else:
            outputs['correction'] = tensor
    
    phase_dev = abs(outputs['phase'].flatten()[0] - keras_phase[i].flatten()[0])
    correction_dev = abs(outputs['correction'].flatten() - keras_correction[i].flatten()).max()
    
    max_phase_dev = max(max_phase_dev, phase_dev)
    max_correction_dev = max(max_correction_dev, correction_dev)

print(f'Max phase deviation:      {max_phase_dev:.6f}')
print(f'Max correction deviation: {max_correction_dev:.6f}')

THRESHOLD = 0.001
if max_phase_dev < THRESHOLD and max_correction_dev < THRESHOLD:
    print(f'\nAll deviations < {THRESHOLD} — TFLite export validated!')
else:
    print(f'\nDeviation exceeds {THRESHOLD} — investigate quantization impact')

## Export TypeScript test fixtures

Save input/output pairs as JSON for validating that the TypeScript
normalization matches Python exactly.

In [ ]:
ts_fixtures = []

for i in range(min(5, len(test_inputs))):
    input_data = test_inputs[i:i+1].astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    
    outputs = {}
    for od in output_details:
        tensor = interpreter.get_tensor(od['index'])
        if tensor.shape[-1] == 1:
            outputs['phase'] = float(tensor.flatten()[0])
        else:
            outputs['correction'] = tensor.flatten().tolist()
    
    ts_fixtures.append({
        'input': test_inputs[i].tolist(),
        'expected_phase': outputs['phase'],
        'expected_correction': outputs['correction'],
    })

fixtures_path = output_dir / 'ts_test_fixtures.json'
with open(fixtures_path, 'w') as f:
    json.dump(ts_fixtures, f, indent=2)

print(f'TypeScript test fixtures saved: {fixtures_path}')
print(f'  {len(ts_fixtures)} test cases')

## Copy to app assets

In [ ]:
import shutil

app_model_dir = Path('../../apps/mobile/assets/models')
app_model_dir.mkdir(parents=True, exist_ok=True)

dest = app_model_dir / 'pose_correction.tflite'
shutil.copy2(tflite_path, dest)
print(f'Copied to {dest}')
print(f'Size: {dest.stat().st_size / 1024:.1f} KB')

---
**Done!** The model is now ready for on-device inference.

## Summary of what was exported

| File | Purpose |
|------|--------|
| `pose_correction.tflite` | Mobile model (float16 quantized) |
| `ts_test_fixtures.json` | Validation data for TypeScript normalization |

## Next steps
1. Integrate `pose_correction.tflite` into the React Native app
2. Implement matching normalization in TypeScript
3. Overlay both user skeleton and correction skeleton on camera feed
4. Add more exercise videos and retrain